# TechOps Intelligence Platform
## Notebook 05 — Tabular Metrics Pipeline

**Goal:** Convert FinTechFlow service metrics into
          searchable anomaly text for diagnosis agent

### Services
- payment-service
- order-service
- transaction-processor
- fraud-detection-service
- postgresql-primary

In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version (compiled):", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


Torch version: 2.5.1+cu121
CUDA available: True
CUDA version (compiled): 12.1
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


In [3]:
import chromadb

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

FTF_METRICS = PROJECT_ROOT / "data/raw/tabular/fintechflow"
EMBEDDINGS  = PROJECT_ROOT / "data/embeddings"

print("Setup complete")
print(f"\nFTF metric files:")
for f in sorted(FTF_METRICS.glob("*.csv")):
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:45} {size:.1f} MB")

    
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Model ready")

chroma_path     = str(EMBEDDINGS / "chroma_db")
client          = chromadb.PersistentClient(path=chroma_path)
logs_collection = client.get_or_create_collection(
    name     = "logs",
    metadata = {"hnsw:space": "cosine"}
)

print(f"logs collection: {logs_collection.count():,} docs")


def get_existing_ids(collection) -> set:
    if collection.count() == 0:
        return set()
    return set(collection.get(include=[])['ids'])


BATCH_SIZE = 64
print("Ready")

Setup complete

FTF metric files:
  fraud_detection_metrics.csv                   0.1 MB
  order_service_metrics.csv                     0.1 MB
  payment_service_metrics.csv                   0.1 MB
  postgresql_primary_metrics.csv                0.1 MB
  transaction_processor_metrics.csv             0.1 MB
Loading embedding model...
Model ready


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


logs collection: 0 docs
Ready


## 1. Anomaly Detection Functions
Z-score based anomaly detection on FTF service metrics
Each anomaly window converted to natural language for RAG

In [4]:
# Unit mapping for FTF metric columns
FTF_UNITS = {
    'cpu_percent'       : ('CPU usage',          'percent'),
    'memory_percent'    : ('memory usage',        'percent'),
    'disk_percent'      : ('disk usage',          'percent'),
    'error_rate_percent': ('error rate',          'percent'),
    'latency_p95_ms'    : ('P95 request latency', 'ms'),
}


def format_value(metric: str, value: float) -> str:
    if metric in FTF_UNITS:
        label, unit = FTF_UNITS[metric]
        return f"{value:.1f} {unit}"
    return f"{value:.2f}"


def classify_pattern(metric: str, value: float) -> str:
    if 'cpu' in metric and value > 90:
        return "CPU saturation causing payment request timeouts"
    if 'memory' in metric and value > 90:
        return "memory exhaustion OOMKilled pod risk"
    if 'disk' in metric and value > 90:
        return "disk critical PostgreSQL WAL or Kafka logs"
    if 'error' in metric and value > 5:
        return "error rate above SLO threshold payment SLO at risk"
    if 'latency' in metric and value > 1000:
        return "latency above 1s circuit breaker may open"
    return "resource anomaly investigate FinTechFlow dashboards"


def detect_anomalies(df, metric_cols,
                     z_threshold=2.0, window_size=5):
    windows = []
    for col in metric_cols:
        if col not in df.columns:
            continue
        series = pd.to_numeric(
            df[col], errors='coerce'
        ).dropna()
        if len(series) < 10:
            continue

        z_scores = np.abs(stats.zscore(series))
        idx      = np.where(z_scores > z_threshold)[0]
        if len(idx) == 0:
            continue

        groups = []
        s = e = idx[0]
        for i in range(1, len(idx)):
            if idx[i] - idx[i-1] <= window_size:
                e = idx[i]
            else:
                groups.append((s, e))
                s = e = idx[i]
        groups.append((s, e))

        for s, e in groups:
            max_z    = float(z_scores[s:e+1].max())
            mean_val = float(series.iloc[s:e+1].mean())
            max_val  = float(series.iloc[s:e+1].max())
            windows.append({
                "metric"    : col,
                "max_zscore": round(max_z, 2),
                "mean_value": round(mean_val, 2),
                "max_value" : round(max_val, 2),
                "duration"  : int(e - s + 1),
                "severity"  : "critical" if max_z > 4.0
                              else "high" if max_z > 3.0
                              else "medium"
            })
    return windows


def anomaly_to_text(anomaly: dict, service: str) -> str:
    metric   = anomaly['metric']
    label    = FTF_UNITS.get(metric, (metric, ''))[0]
    fmt_max  = format_value(metric, anomaly['max_value'])
    fmt_mean = format_value(metric, anomaly['mean_value'])
    pattern  = classify_pattern(metric, anomaly['max_value'])

    return (
        f"FinTechFlow {service} showed a "
        f"{anomaly['severity']} anomaly in {label}. "
        f"Peak was {fmt_max} with mean {fmt_mean} "
        f"over {anomaly['duration']} intervals "
        f"({anomaly['max_zscore']:.1f} std deviations). "
        f"Pattern suggests {pattern}."
    )


print("Anomaly detection functions ready")

Anomaly detection functions ready


## 2. Process FTF Service Metrics
Detect anomalies and embed as natural language

In [5]:
service_files = sorted(FTF_METRICS.glob("*_metrics.csv"))
existing      = get_existing_ids(logs_collection)
total_stored  = 0

print(f"Processing {len(service_files)} FTF service files\n")

for csv_file in service_files:
    service  = csv_file.stem.replace(
        '_metrics', ''
    ).replace('_', '-')

    df       = pd.read_csv(csv_file)
    num_cols = [
        c for c in df.select_dtypes(
            include=[np.number]
        ).columns
        if c not in ('requests_per_min', 'timestamp')
    ]

    anomalies = detect_anomalies(df, num_cols)
    print(f"{service}: {df.shape[0]:,} rows, "
          f"{len(anomalies)} anomaly windows")

    texts, ids, metadatas = [], [], []
    for idx, anomaly in enumerate(anomalies):
        doc_id = (
            f"ftf_metric_{service}"
            f"_{anomaly['metric']}_{idx}"
        )
        if doc_id in existing:
            continue

        text = anomaly_to_text(anomaly, service)
        texts.append(text)
        ids.append(doc_id)
        metadatas.append({
            "source"  : "ftf_metrics",
            "service" : service,
            "metric"  : anomaly['metric'],
            "severity": anomaly['severity'],
            "doc_type": "metric_anomaly"
        })

    if texts:
        for i in range(0, len(texts), BATCH_SIZE):
            bt  = texts[i:i+BATCH_SIZE]
            bi  = ids[i:i+BATCH_SIZE]
            bm  = metadatas[i:i+BATCH_SIZE]
            emb = embedding_model.encode(
                bt, show_progress_bar=False
            )
            logs_collection.upsert(
                documents  = bt,
                embeddings = emb.tolist(),
                metadatas  = bm,
                ids        = bi
            )
        total_stored += len(texts)
        print(f"  Stored: {len(texts)} anomaly windows")
    else:
        print(f"  All already embedded")

print(f"\nTotal stored: {total_stored}")
print(f"Logs total  : {logs_collection.count():,}")

Processing 5 FTF service files

fraud-detection: 2,016 rows, 261 anomaly windows
  Stored: 261 anomaly windows
order-service: 2,016 rows, 178 anomaly windows
  Stored: 178 anomaly windows
payment-service: 2,016 rows, 165 anomaly windows
  Stored: 165 anomaly windows
postgresql-primary: 2,016 rows, 306 anomaly windows
  Stored: 306 anomaly windows
transaction-processor: 2,016 rows, 210 anomaly windows
  Stored: 210 anomaly windows

Total stored: 1120
Logs total  : 1,120


In [6]:
def log_search(query, n=2):
    emb = embedding_model.encode([query]).tolist()
    r   = logs_collection.query(
        query_embeddings = emb,
        n_results        = n,
        include          = ['documents', 'metadatas', 'distances']
    )
    return r


test_queries = [
    "CPU usage spike payment-service",
    "memory exhaustion OOMKilled risk",
    "disk space critical PostgreSQL",
    "error rate above SLO threshold",
    "latency degradation circuit breaker"
]

print("Logs retrieval test\n")
scores = []
for query in test_queries:
    r     = log_search(query, n=1)
    score = 1 - r['distances'][0][0]
    src   = r['metadatas'][0][0].get('service', 'unknown')
    text  = r['documents'][0][0][:100]
    scores.append(score)
    print(f"Query   : {query}")
    print(f"Score   : {score:.3f}  Service: {src}")
    print(f"Result  : {text}")
    print()

print(f"Avg score: {np.mean(scores):.3f}")

Logs retrieval test



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query   : CPU usage spike payment-service
Score   : 0.743  Service: payment-service
Result  : FinTechFlow payment-service showed a medium anomaly in CPU usage. Peak was 47.5 percent with mean 36

Query   : memory exhaustion OOMKilled risk
Score   : 0.355  Service: postgresql-primary
Result  : FinTechFlow postgresql-primary showed a medium anomaly in memory usage. Peak was 80.8 percent with m

Query   : disk space critical PostgreSQL
Score   : 0.371  Service: postgresql-primary
Result  : FinTechFlow postgresql-primary showed a high anomaly in memory usage. Peak was 83.8 percent with mea

Query   : error rate above SLO threshold
Score   : 0.279  Service: fraud-detection
Result  : FinTechFlow fraud-detection showed a medium anomaly in error rate. Peak was 0.1 percent with mean 0.

Query   : latency degradation circuit breaker
Score   : 0.377  Service: order-service
Result  : FinTechFlow order-service showed a critical anomaly in P95 request latency. Peak was 490.3 ms with m

Avg score: 0.

In [8]:
print("=" * 50)
print("NOTEBOOK 05 - TABULAR PIPELINE COMPLETE")
print("=" * 50)
print(f"  logs collection: {logs_collection.count():,} docs")
print(f"  Avg retrieval score: {np.mean(scores):.3f}")


NOTEBOOK 05 - TABULAR PIPELINE COMPLETE
  logs collection: 1,120 docs
  Avg retrieval score: 0.425
